# Random Forest & Bagging: Zero to Hero

Take the unstable model from the last notebook. Build hundreds of them. Average.

> **Prerequisites:**
> [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb) for the shared
> workflow, and [`decision_trees_zero_to_hero.ipynb`](decision_trees_zero_to_hero.ipynb) for
> the base learner. This notebook assumes you know how a tree splits, why it overfits, and
> — most importantly — **why a single tree is unstable**. That instability is not a nuisance
> here. It is the raw material.

---

## Where this comes from

NB-03 Part 3 ended with a measurement: refit a tree on 60 bootstrap resamples of the same
data and you get **five different root splits**, all scoring about the same. Two facts fell
out of that:

- individual trees have **high variance** — they change a lot with the data
- they are **not systematically wrong** — the average accuracy is fine

That exact combination — low bias, high variance, errors that are not all the same error —
is the textbook case for **averaging**. This notebook is what happens when you take that
seriously.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | The bootstrap · why averaging reduces variance (with the formula) · **decorrelation**, the step that makes it a forest · out-of-bag validation derived · why more trees never overfit · the hyperparameters that matter |
| **2. Worked example** | Credit default risk, end to end |
| **3. Feature importance, done properly** | The part everyone gets wrong — MDI vs permutation vs drop-column, and what correlated features do to all three |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | Forest-specific errors and a checklist |

## The one-paragraph summary

**Bagging** fits many copies of a model on **bootstrap resamples** of the training data and
averages them. Averaging cancels the part of each model's error that is independent across
models, so variance falls while bias stays put — which is why it works on high-variance,
low-bias learners like unpruned trees and does nothing for a linear model. A **random
forest** adds one further trick: at every split, each tree may only consider a **random
subset of the features**. That deliberately makes the individual trees *worse* and *less
correlated with each other*, and because the correlation between trees is what limits how far
averaging can go, the ensemble comes out better.

---
# Part 0 - Setup

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    StratifiedKFold, GridSearchCV, RandomizedSearchCV, learning_curve,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, mean_absolute_error,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110

print("ready | numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 - Theory from zero

1. The bootstrap: sampling with replacement
2. Bagging: why averaging reduces variance — with the formula
3. Decorrelation: the step that turns bagging into a *forest*
4. Out-of-bag: free validation, derived
5. Why more trees never overfit
6. The hyperparameters that actually matter

## 1.1 The bootstrap

Draw $n$ samples from a dataset of size $n$, **with replacement**. Some rows appear twice or
three times; some do not appear at all. That is a *bootstrap resample* — a plausible
alternative version of the dataset you might have collected.

The key quantity: what fraction of the original rows make it in?

A given row is missed on one draw with probability $1 - 1/n$. Over $n$ independent draws:

$$ P(\text{row never drawn}) = \left(1 - \frac{1}{n}\right)^{n} \;\xrightarrow[n \to \infty]{}\; e^{-1} \approx 0.368 $$

So each resample contains about **63.2%** of the unique rows, and leaves **36.8%** out. Both
numbers matter: the first is why the trees differ, the second is free validation data (1.4).

In [ ]:
rng = np.random.default_rng(0)

print(f"{'n':>8} {'mean unique fraction':>22} {'1 - 1/e':>10}")
print("-" * 44)
for n in [10, 50, 200, 1000, 10_000]:
    fractions = [len(np.unique(rng.integers(0, n, n))) / n for _ in range(400)]
    print(f"{n:>8} {np.mean(fractions):>22.4f} {1 - np.exp(-1):>10.4f}")

print()
print("It converges fast - by n=200 it is already within 0.003 of the limit.")
print()
# What one resample actually looks like.
data = np.arange(10)
sample = rng.integers(0, 10, 10)
counts = np.bincount(sample, minlength=10)
print("one bootstrap resample of the rows 0-9:")
print("  drawn      :", sorted(sample.tolist()))
print("  times drawn:", counts.tolist())
print("  left out   :", np.where(counts == 0)[0].tolist(), " <- these are 'out-of-bag'")

## 1.2 Bagging: why averaging reduces variance

**B**ootstrap **agg**regat**ing**: fit a model on each of $B$ bootstrap resamples and average
the predictions (or vote, for classification).

Why should that help? Here is the whole argument in one formula. Suppose each model's
prediction at some point is a random variable with variance $\sigma^2$, and any two models
have pairwise correlation $\rho$. The variance of their average is:

$$ \operatorname{Var}\!\left(\frac{1}{B}\sum_{b=1}^{B} f_b\right) \;=\; \rho\,\sigma^2 \;+\; \frac{1-\rho}{B}\,\sigma^2 $$

Read the two terms, because the entire design of a random forest is in them:

- The **second term vanishes** as $B \to \infty$. More trees always help, and never hurt.
- The **first term does not**. It is a floor set by $\rho$, the correlation between models.
  No number of trees gets you below $\rho\sigma^2$.

**So the way to improve a large ensemble is not more trees — it is less correlated trees.**
That is 1.3, and it is the entire difference between bagging and a random forest.

Note what averaging does *not* change: **bias**. If every model is wrong in the same
direction, averaging preserves that error exactly. This is why bagging is applied to
**low-bias, high-variance** learners — deep unpruned trees — and does essentially nothing for
a linear model.

In [ ]:
# Verify the formula empirically, then use it to make the point.
def average_variance(rho, sigma2, B):
    """The theoretical variance of the mean of B correlated predictors."""
    return rho * sigma2 + (1 - rho) * sigma2 / B


rng = np.random.default_rng(1)
sigma2, trials = 1.0, 4000

print(f"{'rho':>6} {'B':>6} {'empirical var':>15} {'formula':>10}")
print("-" * 42)
for rho in [0.0, 0.3, 0.8]:
    for B in [1, 10, 100]:
        # B predictors with pairwise correlation rho: shared component + private component.
        shared = rng.normal(0, np.sqrt(rho * sigma2), trials)
        private = rng.normal(0, np.sqrt((1 - rho) * sigma2), (trials, B))
        avg = shared + private.mean(axis=1)
        print(f"{rho:>6.1f} {B:>6} {avg.var():>15.4f} {average_variance(rho, sigma2, B):>10.4f}")

print()
print("Now the consequence, at B = 500 trees:")
for rho in [0.0, 0.2, 0.5, 0.9]:
    v = average_variance(rho, 1.0, 500)
    print(f"  rho={rho:<4} -> variance {v:.4f}   "
          f"({v/1.0:.1%} of a single model's variance)")
print()
print("At rho=0.9 you keep 90% of the variance no matter how many trees you build.")
print("At rho=0.2 you keep 20%. THAT is the lever, and 1.3 is how you pull it.")

In [ ]:
# The bias claim, checked: bagging a high-variance learner vs a low-variance one.
X_demo, y_demo = make_classification(n_samples=1500, n_features=15, n_informative=6,
                                     n_redundant=4, random_state=RANDOM_STATE)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    X_demo, y_demo, test_size=0.3, stratify=y_demo, random_state=RANDOM_STATE)
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

print(f"{'base learner':<34} {'alone':>16} {'bagged x100':>16}")
print("-" * 68)
for label, base in [
    ("deep tree (high variance)", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ("depth-2 stump (high BIAS)", DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)),
    ("logistic regression (low var)", LogisticRegression(max_iter=2000)),
]:
    alone = cross_val_score(base, Xd_tr, yd_tr, cv=cv, scoring="roc_auc")
    bagged = cross_val_score(
        BaggingClassifier(base, n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        Xd_tr, yd_tr, cv=cv, scoring="roc_auc")
    print(f"{label:<34} {alone.mean():>8.4f} +/-{alone.std():<5.3f} "
          f"{bagged.mean():>8.4f} +/-{bagged.std():<5.3f}")

print()
print("The bottom row is the cleanest result: bagging logistic regression changes NOTHING.")
print("Refit a linear model on a bootstrap resample and you get nearly the same")
print("coefficients, so all 100 copies are near-identical and averaging has nothing to")
print("cancel. This is Breiman's (1996) actual criterion - bagging helps UNSTABLE learners.")
print()
print("The stump is the subtle row, and it is worth being precise about. Bagging does help")
print("it - which two splits a depth-2 tree picks does vary across resamples, so there IS")
print("variance to remove. But look at where it LANDS, not how far it moved: even bagged,")
print("it stays far below the bagged deep tree. The variance came out; the bias stayed in,")
print("and the bias is what caps it.")
print()
print("That is the rule to carry: averaging can only ever recover the variance component.")
print("Give it a base learner that is too rigid to capture the structure and the ceiling is")
print("set before you start - which is exactly why forests use UNPRUNED trees (1.6).")

## 1.3 Decorrelation: the step that makes it a *forest*

Bagging trees has a problem. Every tree sees ~63% of the same rows and **all** of the
features, so if one feature is clearly the strongest predictor, nearly every tree picks it at
the root. The trees end up structurally similar — high $\rho$ — and 1.2 says that caps how
much averaging can buy.

Breiman's fix in *Random Forests* (2001) is almost crude: **at every split, allow the tree to
consider only a random subset of the features** (`max_features`). A tree that cannot see the
dominant feature at some node is forced to find a different, weaker split — and so the trees
diverge.

This makes each individual tree **worse**. That is the point. You are trading a little
accuracy per tree for a lot of decorrelation, and 1.2 says the ensemble wins that trade.

| `max_features` | Effect |
|---|---|
| `None` (all) | This is plain **bagging**. Highest $\rho$. |
| `"sqrt"` | $\sqrt{p}$ features. The **classification default**, and usually right. |
| `1.0` / `None` for regression | sklearn's regression default; often worth lowering to ~0.3 |
| `1` | One random feature per split — maximal decorrelation, very weak trees |

In [ ]:
X_rf, y_rf = make_classification(n_samples=3000, n_features=20, n_informative=8,
                                 n_redundant=6, weights=[0.85, 0.15],
                                 random_state=RANDOM_STATE)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_rf, y_rf, test_size=0.3, stratify=y_rf, random_state=RANDOM_STATE)

print(f"{'max_features':<16} {'mean tree correlation':>22} {'CV ROC-AUC':>13}")
print("-" * 54)
for mf, note in [(None, "= plain bagging"), ("sqrt", "= sklearn default"),
                 (0.3, ""), (1, "= one feature per split")]:
    forest = RandomForestClassifier(n_estimators=100, max_features=mf,
                                    random_state=RANDOM_STATE, n_jobs=-1).fit(Xr_tr, yr_tr)
    # Correlation between the individual trees' predicted probabilities.
    preds = np.array([t.predict_proba(Xr_te)[:, 1] for t in forest.estimators_])
    corr = np.corrcoef(preds)
    rho = (corr.sum() - len(corr)) / (len(corr) * (len(corr) - 1))

    auc = cross_val_score(RandomForestClassifier(n_estimators=100, max_features=mf,
                                                 random_state=RANDOM_STATE, n_jobs=-1),
                          Xr_tr, yr_tr, cv=cv, scoring="roc_auc").mean()
    print(f"{str(mf):<16} {rho:>22.4f} {auc:>13.4f}  {note}")

print()
print("Read the two columns together - this is the whole design of a random forest.")
print()
print("  max_features=None is bagging: the HIGHEST tree correlation, and it loses.")
print("  max_features='sqrt' cuts correlation and WINS on accuracy.")
print("  max_features=1 cuts correlation furthest of all - and loses again.")
print()
print("That last row is the part people miss. Decorrelation is not free: restrict the")
print("features far enough and each tree becomes too weak to be worth averaging. The best")
print("setting is a compromise between tree STRENGTH and tree INDEPENDENCE, which is")
print("exactly what the rho*sigma^2 + (1-rho)*sigma^2/B formula predicts.")

### Extremely Randomised Trees, for contrast

`ExtraTreesClassifier` pushes the same idea one step further: it does not search for the best
threshold on each candidate feature, it picks the threshold **at random** and keeps the best
of those. Even more decorrelation, even weaker trees, and (by default) no bootstrapping at
all.

Worth trying whenever a forest is close but not quite good enough — it costs one line and is
usually faster to fit, because the expensive threshold search is skipped.

In [ ]:
print(f"{'model':<34} {'CV ROC-AUC':>13} {'fit time (s)':>14}")
print("-" * 62)
for label, model in [
    ("bagged trees (max_features=None)",
     RandomForestClassifier(n_estimators=200, max_features=None,
                            random_state=RANDOM_STATE, n_jobs=-1)),
    ("random forest (max_features=sqrt)",
     RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    ("extra trees",
     ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
]:
    res = cross_validate(model, Xr_tr, yr_tr, cv=cv, scoring="roc_auc")
    print(f"{label:<34} {res['test_score'].mean():>13.4f} {res['fit_time'].mean():>14.2f}")

print()
print("Extra trees is competitive and cheaper to fit. It is not universally better - on")
print("noisy data the random thresholds can cost too much accuracy per tree - but it is a")
print("free thing to try, and it is the same idea taken to its conclusion.")

## 1.4 Out-of-bag: validation you already paid for

From 1.1: each tree leaves out about **36.8%** of the rows. Those rows are *out-of-bag* for
that tree — never seen during its fitting — so they are legitimate held-out data **for that
tree**.

Aggregate across the forest: for each training row, collect the votes of only the trees that
did not see it, and score those. That is the **OOB score**, and it is a genuine
generalisation estimate that costs nothing extra.

In [ ]:
# With few trees, some rows are out-of-bag for NO tree at all, and sklearn warns.
# We capture that warning and report it as a column instead of letting raw stderr
# into the notebook - the warning is a real signal here, not noise.
import warnings

print(f"{'n_estimators':>13} {'OOB score':>11} {'test score':>12} {'difference':>12}   note")
print("-" * 68)
for n in [10, 50, 200, 1000]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        forest = RandomForestClassifier(n_estimators=n, oob_score=True,
                                        random_state=RANDOM_STATE,
                                        n_jobs=-1).fit(Xr_tr, yr_tr)
    note = ("sklearn warned: too few trees"
            if any("OOB" in str(w.message) for w in caught) else "")
    test = forest.score(Xr_te, yr_te)
    print(f"{n:>13} {forest.oob_score_:>11.4f} {test:>12.4f} "
          f"{test - forest.oob_score_:>+12.4f}   {note}")

print()
print("Two things to notice.")
print()
print("1. OOB is consistently BELOW the test score, and that is not a bug. Each OOB")
print("   prediction is made by only the ~37% of trees that excluded that row, so it comes")
print("   from a SMALLER forest than the one you will actually deploy. A smaller forest is")
print("   a slightly worse forest, so OOB is mildly PESSIMISTIC.")
print()
print("2. The gap shrinks as n_estimators grows, because 37% of 1000 trees is still a big")
print("   forest. With few trees, OOB is both noisy and clearly biased downward - the")
print("   warning sklearn emits below ~50 trees is telling you exactly this.")
print()
print("Use OOB as a cheap sanity check and for tuning on large data where CV is expensive.")
print("For a number you will report, still use proper cross-validation or a test set.")

## 1.5 Why more trees never overfit

This is the single most useful practical property of a random forest, and it surprises people
who have internalised "more capacity → overfitting".

Adding trees does **not** add capacity. Each tree is fitted **independently** of the others —
no tree ever sees another's errors. Adding trees only reduces the Monte-Carlo noise in the
average, driving the $\frac{1-\rho}{B}\sigma^2$ term towards zero. The estimator converges;
it does not stretch.

**Boosting is the opposite.** Each new tree is fitted to the *residual errors* of the ones
before it, so the ensemble keeps bending towards the training data and eventually fits its
noise. More boosting rounds genuinely can, and do, overfit — which is why boosting needs
early stopping and forests do not.

In [ ]:
# Noisy data (flip_y=0.25), so the difference is unmistakable.
X_noisy, y_noisy = make_classification(n_samples=1200, n_features=15, n_informative=5,
                                       flip_y=0.25, random_state=RANDOM_STATE)
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    X_noisy, y_noisy, test_size=0.3, stratify=y_noisy, random_state=RANDOM_STATE)

counts = [1, 10, 50, 200, 800]
rf_scores, gb_scores = [], []
for n in counts:
    rf_scores.append(RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE,
                                            n_jobs=-1).fit(Xn_tr, yn_tr).score(Xn_te, yn_te))
    gb_scores.append(GradientBoostingClassifier(n_estimators=n, learning_rate=0.3, max_depth=5,
                                                random_state=RANDOM_STATE
                                                ).fit(Xn_tr, yn_tr).score(Xn_te, yn_te))

print(f"{'n_estimators':>13} {'random forest':>15} {'gradient boosting':>19}")
print("-" * 50)
for n, r, g in zip(counts, rf_scores, gb_scores):
    print(f"{n:>13} {r:>15.4f} {g:>19.4f}")

plt.plot(counts, rf_scores, marker="o", label="random forest")
plt.plot(counts, gb_scores, marker="s", label="gradient boosting (lr=0.3, depth 5)")
plt.xscale("log"); plt.xlabel("number of trees"); plt.ylabel("test accuracy")
plt.title("More trees: a forest converges, boosting can turn back")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

print()
print("The forest climbs and then flattens - it converges to its ceiling and stays there.")
print("Boosting peaks and then DECLINES, because every additional tree is still chasing")
print("residuals that are now mostly noise.")
print()
print("Practical rule: set n_estimators as high as your compute budget allows and stop")
print("thinking about it. It is one of the very few hyperparameters in ML that you cannot")
print("set too high - only too low. (For boosting it is the most dangerous one.)")

## 1.6 The hyperparameters that actually matter

Forests are famously forgiving — the defaults are good, and this is one of the few models
where "just fit it" is a defensible first move. In rough order of what is worth your time:

| Parameter | Default | What it does | Worth tuning? |
|---|---|---|---|
| `n_estimators` | 100 | Number of trees | **No** — set it as high as you can afford (1.5) |
| `max_features` | `"sqrt"` (clf), `1.0` (reg) | Features considered per split — **the decorrelation dial** | **Yes.** The most important one, especially for regression |
| `min_samples_leaf` | 1 | Minimum samples in a leaf | Yes on noisy data; raise it to 5–20 |
| `max_depth` | `None` | Depth cap | Usually leave unlimited — see below |
| `class_weight` | `None` | Rare-class weighting | On imbalanced data |
| `bootstrap` | `True` | Resample rows | Rarely; `False` makes it non-bagged |
| `n_jobs` | `None` | Parallelism | **Always set `-1`** — trees are embarrassingly parallel |

### Why you usually leave the trees unpruned

This inverts NB-03's advice, and the reason is 1.2: **averaging fixes variance but not bias.**

- A deep unpruned tree is low-bias, high-variance → exactly what averaging repairs.
- A shallow pruned tree is higher-bias → the ensemble inherits that bias and cannot remove it.

So in a forest you *want* the individual trees overfitting. Prune only when the data is very
noisy or you need to control memory/latency.

In [ ]:
print("does pruning the base trees help or hurt the forest?\n")
print(f"{'max_depth':>10} {'single tree':>14} {'forest of 300':>16}")
print("-" * 44)
for depth in [2, 4, 8, None]:
    tree = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE),
                           Xr_tr, yr_tr, cv=cv, scoring="roc_auc").mean()
    forest = cross_val_score(RandomForestClassifier(n_estimators=300, max_depth=depth,
                                                    random_state=RANDOM_STATE, n_jobs=-1),
                             Xr_tr, yr_tr, cv=cv, scoring="roc_auc").mean()
    print(f"{str(depth):>10} {tree:>14.4f} {forest:>16.4f}")

print()
print("For the single tree, depth is a real trade-off. For the forest, deeper is better or")
print("neutral all the way to unlimited - the averaging is absorbing the variance that")
print("pruning would otherwise have to control.")
print()
print("This is the clearest possible illustration that bagging is a VARIANCE mechanism:")
print("removing variance from the base learner (by pruning) removes the ensemble's job,")
print("while the bias you add by pruning is permanent.")

---
# Part 2 - Worked example: credit default risk

Predict which loan applicants will default. ~15% do, so it is imbalanced; several features
are **redundant** (linear combinations of the informative ones), which is deliberate — that
is what makes Part 3 bite.

The workflow itself is Foundations Part 1; we move quickly through it and spend the time on
what is forest-specific.

In [ ]:
X_credit, y_credit = make_classification(
    n_samples=6000, n_features=16,
    n_informative=6,        # only 6 features genuinely drive the label
    n_redundant=5,          # 5 more are linear combinations of those 6
    n_repeated=0,
    weights=[0.85, 0.15],   # 15% default rate
    flip_y=0.03,            # a little label noise, as in real credit data
    class_sep=0.9,
    random_state=RANDOM_STATE,
)
credit_features = [f"f{i:02d}" for i in range(16)]
X_credit = pd.DataFrame(X_credit, columns=credit_features)

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_credit, y_credit, test_size=0.25, stratify=y_credit, random_state=RANDOM_STATE)

print(f"{len(X_credit)} applicants, {X_credit.shape[1]} features")
print(f"default rate: {y_credit.mean():.3f}")
print(f"train {len(Xc_tr)}  test {len(Xc_te)}")
print("\nby construction: 6 informative, 5 redundant (combinations of those 6), 5 pure noise")

dummy = DummyClassifier(strategy="most_frequent").fit(Xc_tr, yc_tr)
print(f"\nbaseline (never default): accuracy {dummy.score(Xc_te, yc_te):.4f}, "
      f"recall 0.0000, ROC-AUC 0.5000")

In [ ]:
# No scaler, no imputer - trees need neither. This IS the pipeline.
forest = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced_subsample",   # see the note below
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit(Xc_tr, yc_tr)

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_validate(
    RandomForestClassifier(n_estimators=500, max_features="sqrt", min_samples_leaf=2,
                           class_weight="balanced_subsample",
                           random_state=RANDOM_STATE, n_jobs=-1),
    Xc_tr, yc_tr, cv=cv, scoring=["roc_auc", "average_precision"])

print(f"OOB ACCURACY (free)  : {forest.oob_score_:.4f}   <- accuracy, not AUC; not comparable to the rows below")
print(f"CV ROC-AUC           : {scores['test_roc_auc'].mean():.4f} "
      f"(+/- {scores['test_roc_auc'].std():.4f})")
print(f"CV PR-AUC            : {scores['test_average_precision'].mean():.4f} "
      f"(base rate {yc_tr.mean():.4f})")
print()
print("class_weight='balanced_subsample' re-weights within each tree's bootstrap sample")
print("rather than once globally - the right variant for a forest, since every tree sees a")
print("different class balance. As in NB-02, it improves recall at the default threshold and")
print("distorts the probabilities; check calibration before doing expected-value maths.")

In [ ]:
test_proba = forest.predict_proba(Xc_te)[:, 1]
test_pred = forest.predict(Xc_te)

tn, fp, fn, tp = confusion_matrix(yc_te, test_pred).ravel()
print("at the default 0.5 threshold")
print(f"                    predicted repay   predicted default")
print(f"  actually repay  {tn:>17}   {fp:>17}")
print(f"  actually default{fn:>17}   {tp:>17}")
print()
print(classification_report(yc_te, test_pred, target_names=["repay", "default"], digits=3))
print(f"ROC-AUC {roc_auc_score(yc_te, test_proba):.4f}   "
      f"PR-AUC {average_precision_score(yc_te, test_proba):.4f}   "
      f"(base rate {yc_te.mean():.4f})")
print(f"Brier   {brier_score_loss(yc_te, test_proba):.4f}")
print()
print("Everything about thresholds and calibration from NB-02 Part 3 applies unchanged here.")
print("A forest's predict_proba is a VOTE SHARE, not a fitted probability, and it tends to")
print("be pulled toward the middle - forests are usually less well calibrated than an")
print("unweighted logistic regression. Wrap in CalibratedClassifierCV if you need honest")
print("probabilities.")

In [ ]:
# Is the forest actually earning its complexity here?
print(f"{'model':<34} {'CV ROC-AUC':>13} {'fit time (s)':>14}")
print("-" * 62)
for label, model in [
    ("single tree (depth 6)", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
    ("logistic regression", make_pipeline(StandardScaler(),
                                          LogisticRegression(max_iter=2000))),
    ("random forest (500)", RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                                   random_state=RANDOM_STATE, n_jobs=-1)),
]:
    res = cross_validate(model, Xc_tr, yc_tr, cv=cv, scoring="roc_auc")
    print(f"{label:<34} {res['test_score'].mean():>7.4f} +/-{res['test_score'].std():<5.3f}"
          f" {res['fit_time'].mean():>13.2f}")

print()
print("Run this comparison on every project. The forest should beat the single tree")
print("comfortably (that is the whole thesis of this notebook). Whether it beats logistic")
print("regression tells you whether there is real non-linear structure to find - and if it")
print("does not, you have just saved yourself an unreadable model.")

---
# Part 3 - Feature importance, done properly

This is the part practitioners get wrong most often, and forests make it worse by being
*good* at reporting numbers that look authoritative.

Three different methods answer three genuinely different questions:

| Method | What it measures | Computed on | Cost |
|---|---|---|---|
| **MDI** (`feature_importances_`) | Total impurity reduction from splits on this feature | **Training** data | Free |
| **Permutation** | Score drop when this column is shuffled | Any data — use **test** | Cheap |
| **Drop-column** | Score drop when the model is **refitted without** it | Any data | Expensive (refit per feature) |

They disagree, and the disagreements are informative rather than annoying.

NB-03 Challenge 2 showed MDI's **cardinality bias** (continuous and high-cardinality noise
features get inflated importance). Here we tackle the other big failure, which forests do not
fix and which is far more common in real data: **correlated features**.

In [ ]:
# Three near-copies of one underlying driver, plus one weaker independent driver.
rng = np.random.default_rng(0)
n_imp = 3000
driver = rng.normal(size=n_imp)

X_imp = pd.DataFrame({
    "signal_a": driver + rng.normal(0, 0.15, n_imp),   # three noisy measurements
    "signal_b": driver + rng.normal(0, 0.15, n_imp),   #  of the SAME thing
    "signal_c": driver + rng.normal(0, 0.15, n_imp),
    "other":    rng.normal(size=n_imp),                # a genuinely separate, weaker driver
    "noise":    rng.normal(size=n_imp),                # nothing at all
})
logit = 2.0 * driver + 0.8 * X_imp["other"]
y_imp = (rng.random(n_imp) < 1 / (1 + np.exp(-logit))).astype(int)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_imp, y_imp, test_size=0.3, stratify=y_imp, random_state=RANDOM_STATE)

print("correlation between the three signal columns:")
print(X_imp[["signal_a", "signal_b", "signal_c"]].corr().round(3).to_string())
print("\nTRUTH: the label depends on `driver` (measured by all three signals) and on `other`.")
print("       `noise` does nothing.")

In [ ]:
single = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE).fit(Xi_tr, yi_tr)
forest_imp = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                    n_jobs=-1).fit(Xi_tr, yi_tr)

print("MDI: single tree vs forest\n")
print(pd.DataFrame({
    "single_tree_MDI": single.feature_importances_,
    "forest_MDI": forest_imp.feature_importances_,
}, index=X_imp.columns).round(4).to_string())

print()
print("The single tree's answer is essentially arbitrary: one of the three identical signals")
print("wins the root split and takes almost all the credit, the other two get scraps. Refit")
print("on a resample (NB-03 Part 3) and a different one wins.")
print()
print("The forest spreads the credit almost EVENLY across the three, because each tree sees")
print("a different random feature subset and so different signals get their turn. That is a")
print("real improvement in honesty - the three columns genuinely are equally informative.")
print()
print("But now look at `noise`, which by construction does nothing at all. It still receives")
print("non-trivial MDI, because MDI is measured on TRAINING data where random splits look")
print("useful. MDI is never zero for a feature the trees were allowed to use.")

In [ ]:
perm = permutation_importance(forest_imp, Xi_te, yi_te, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="roc_auc")

comparison = pd.DataFrame({
    "forest_MDI": forest_imp.feature_importances_,
    "permutation_test": perm.importances_mean,
    "permutation_std": perm.importances_std,
}, index=X_imp.columns)
print(comparison.round(4).to_string())

print()
print("Permutation importance correctly reports `noise` at ~0 - shuffling it costs nothing,")
print("because it carries nothing. That is the fix for MDI's training-data optimism.")
print()
print("But look at the three signals. Each gets only a SMALL importance, far less than you")
print("would expect from something that drives the label. Why?")
print()
print("Because when you shuffle signal_a, the model simply reads the answer off signal_b and")
print("signal_c, which are still intact. Permutation importance asks 'what if I lose this")
print("column, keeping everything else?' - and with redundant columns, the answer is")
print("'almost nothing', for EVERY one of them individually.")
print()
print("Read naively, this table says the three most important features barely matter.")

In [ ]:
# The drop-column test: refit without the feature. Expensive, and the honest answer.
def drop_column_importance(model_factory, X_tr, y_tr, X_te, y_te, groups):
    base_model = model_factory().fit(X_tr, y_tr)
    base = roc_auc_score(y_te, base_model.predict_proba(X_te)[:, 1])
    rows = []
    for label, cols in groups.items():
        m = model_factory().fit(X_tr.drop(columns=cols), y_tr)
        auc = roc_auc_score(y_te, m.predict_proba(X_te.drop(columns=cols))[:, 1])
        rows.append({"dropped": label, "AUC_without": auc, "loss": base - auc})
    return base, pd.DataFrame(rows)


factory = lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                         n_jobs=-1)
base_auc, drops = drop_column_importance(
    factory, Xi_tr, yi_tr, Xi_te, yi_te,
    groups={
        "signal_a": ["signal_a"],
        "signal_b": ["signal_b"],
        "signal_c": ["signal_c"],
        "other": ["other"],
        "noise": ["noise"],
        "ALL THREE signals": ["signal_a", "signal_b", "signal_c"],
    })

print(f"full-model test ROC-AUC: {base_auc:.4f}\n")
print(drops.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

group_loss = float(drops.loc[drops["dropped"] == "ALL THREE signals", "loss"].iloc[0])
solo_losses = drops[drops["dropped"].isin(["signal_a", "signal_b", "signal_c"])]["loss"]
noise_loss = float(drops.loc[drops["dropped"] == "noise", "loss"].iloc[0])

print()
print("THERE it is. Dropping any ONE of the three signals costs at most "
      f"{solo_losses.max():.4f} AUC -")
print(f"the other two cover for it. Dropping ALL THREE costs {group_loss:.4f}.")
print()
print("Note also that `noise` shows a loss of "
      f"{noise_loss:.4f} - comparable to, or larger than, the")
print("individual signals. That is not a finding either: a refitted forest is stochastic,")
print("so single-feature drop-column differences of this size are inside the noise floor.")
print("Repeat each drop with several seeds before believing any small number.")
print()
print("So the honest statement is not about individual features at all:")
print("  'signal_a is unimportant'                  -> WRONG, and dangerous")
print("  'signal_a is redundant GIVEN b and c'      -> right")
print("  'the driver those three measure is the single most important thing' -> the truth")
print()
print("This is why you must group correlated features before interpreting importance. Any")
print("per-feature number, from any method, will understate a driver that is measured")
print("several ways - and real datasets are full of those.")

### What to actually do

1. **Cluster correlated features first**, then report importance **per group**. A hierarchical
   clustering on Spearman correlation is the standard recipe (sklearn's docs have a worked
   example under "Permutation Importance with Multicollinear Features").
2. **Use permutation importance on held-out data** as your default per-feature number — it is
   cheap and it is not fooled by training-set noise.
3. **Use drop-column for the features that matter**, when the answer will drive a decision
   (dropping a column from a data contract, dropping a sensor). It is the only method that
   answers "what happens if I genuinely no longer have this?".
4. **Never report MDI on an unpruned model** without saying so. It is training-set impurity,
   biased towards high-cardinality features, and it never returns zero.
5. **None of these measure causation.** A feature can be important because it is a
   *consequence* of the target (leakage), a proxy, or a confounder.

In [ ]:
# Grouping correlated features, the practical recipe.
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr

corr = spearmanr(Xi_tr).correlation
corr = (corr + corr.T) / 2                 # enforce exact symmetry
np.fill_diagonal(corr, 1.0)
distance = 1 - np.abs(corr)
linkage = hierarchy.ward(squareform(distance, checks=False))

for threshold in [0.3, 0.5]:
    cluster_ids = hierarchy.fcluster(linkage, threshold, criterion="distance")
    grouped = {}
    for name, cid in zip(X_imp.columns, cluster_ids):
        grouped.setdefault(cid, []).append(name)
    print(f"threshold {threshold}: {list(grouped.values())}")

print()
print("The three signals fall into one cluster; `other` and `noise` stand alone. Now report")
print("importance for the CLUSTER (drop all its members together), which is what the")
print(f"drop-column table above already showed: that group is worth {group_loss:.3f} AUC,")
print("while no individual member of it looked worth more than a rounding error.")
print()
print("A practical shortcut when you have many features: keep one representative per cluster")
print("and refit. You usually lose almost nothing, and the resulting importances are")
print("interpretable because the redundancy is gone.")

---
# Part 4 - Tough questions

---

### Q1. Derive the 36.8%. Why does each bootstrap sample leave out about that fraction, and what are *both* numbers used for?

<details><summary>Answer</summary>

A specific row is **not** picked on one draw with probability $1 - \frac{1}{n}$. Draws are
independent, and there are $n$ of them, so:

$$ P(\text{never drawn}) = \left(1-\tfrac{1}{n}\right)^{n} \to e^{-1} \approx 0.368 $$

using $\lim_{n\to\infty}(1-1/n)^n = e^{-1}$. Part 1.1 confirms it converges by about $n=200$.

**Both numbers do work:**

- The **63.2% included** is what makes the trees different from each other. Each tree sees a
  different ~63% of the rows, so it grows differently — that is the source of the diversity
  that averaging exploits.
- The **36.8% excluded** is genuinely held-out data *for that tree*, which gives you the
  **OOB score** for free (1.4).

A detail worth knowing: 63.2% is the fraction of **unique** rows, not the sample size. The
resample still has $n$ rows — some appear two or three times. Those duplicates matter: they
are effectively sample weights, and they are part of why the trees differ.

</details>

---

### Q2. Bagging reduces variance. Write down the formula that says so, and explain why more trees eventually stops helping.

<details><summary>Answer</summary>

For $B$ predictors each with variance $\sigma^2$ and pairwise correlation $\rho$:

$$ \operatorname{Var}\!\left(\frac{1}{B}\sum_b f_b\right) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2 $$

- The **second term** is the averaging benefit. It falls as $1/B$, so it is large at first and
  quickly negligible — this is why going from 10 to 100 trees helps a lot and 500 to 5000
  barely registers.
- The **first term, $\rho\sigma^2$, is a floor.** It does not depend on $B$ at all. No number
  of trees gets you below it.

So more trees hits diminishing returns against a hard floor set by how **correlated** the
trees are. That is precisely why a random forest attacks $\rho$ rather than just adding trees
(Q3).

**And note what is absent from the formula: bias.** Averaging does not touch it. If every tree
is wrong in the same direction, the ensemble is wrong in that direction too — which is why
bagging a high-bias learner (a stump, a linear model) accomplishes almost nothing, as Part 1.2
measures.

</details>

---

### Q3. `max_features='sqrt'` makes each individual tree *worse*. Why is that a good idea, and why is `max_features=1` not even better?

<details><summary>Answer</summary>

From Q2, the ensemble's variance floor is $\rho\sigma^2$. Restricting the features available
at each split forces trees to use different splits, which **lowers $\rho$** — and lowering the
floor is the only way to improve a large ensemble.

The cost is that each tree, denied its best feature at some nodes, is individually less
accurate. You are trading $\sigma^2$ (up a little) for $\rho$ (down a lot), and the product
$\rho\sigma^2$ falls.

**Why maximal decorrelation is not optimal:** at `max_features=1` each split is on a single
random feature, so the trees are nearly independent — but they are also so weak that
$\sigma^2$ has grown more than $\rho$ has shrunk. Part 1.3 measures exactly this: tree
correlation drops from 0.43 (bagging) to 0.34 (`sqrt`) to 0.18 (`max_features=1`), while CV
AUC goes **up** then **down**.

So `max_features` is a genuine bias-variance dial between **tree strength** and **tree
independence**, and $\sqrt{p}$ is an empirically good default rather than a theorem.

For **regression** sklearn defaults to `max_features=1.0` (all features), which is plain
bagging — often worth lowering to ~0.3. This is the single most valuable forest
hyperparameter to tune.

</details>

---

### Q4. Your OOB score is 0.93 and your test score is 0.945. Should you be worried?

<details><summary>Answer</summary>

**No — that is the expected direction**, and Part 1.4 reproduces it.

Each OOB prediction for a row uses only the trees that *excluded* that row: about 37% of the
forest. So the OOB score measures a **smaller forest** than the one you will deploy, and a
smaller forest is slightly worse. **OOB is mildly pessimistic by construction.**

The gap shrinks as `n_estimators` grows, because 37% of 1000 trees is still a large forest.
Part 1.4 shows it narrowing from +0.024 at 10 trees to +0.007 at 1000.

**When you SHOULD worry:** if OOB is much *higher* than test. That points at leakage, a
distribution shift between train and test, or a broken split — because there is no honest
mechanism that makes OOB optimistic.

**Where OOB genuinely helps:** cheap model selection on large data, where 5-fold CV would cost
five times as much. Where it does not: as a number you report. Use CV or a held-out set for
that, and note that OOB assumes i.i.d. rows — it is invalid for grouped or time-series data,
in exactly the way a random split is (Foundations Part 3).

</details>

---

### Q5. Why can you not overfit a random forest by adding trees, when adding boosting rounds definitely can?

<details><summary>Answer</summary>

**Because the trees are fitted independently.** No tree in a forest ever sees another tree's
errors. Adding trees only reduces the Monte-Carlo noise in the average — the
$\frac{1-\rho}{B}\sigma^2$ term of Q2 — driving it towards zero. The estimator **converges**
to a fixed limit; it does not gain capacity.

**Boosting is sequential.** Each tree is fitted to the *residuals* of the current ensemble, so
every round bends the model further towards the training data. Once the real signal is
exhausted, further rounds fit noise — and test error turns back up. Part 1.5 shows this
directly on noisy data: the forest climbs and plateaus, boosting peaks and declines.

**Practical consequence:** `n_estimators` is one of the very few hyperparameters you cannot
set too high — only too low, and too expensive. Set it as large as your compute allows and
stop thinking about it. For boosting it is the most dangerous parameter and requires early
stopping.

**The nuance to be precise about:** a forest of unpruned trees *does* fit the training data
perfectly (training accuracy 1.0 in Part 1.5). It is not "immune to overfitting" in general —
its generalisation gap is set by tree depth, data size and noise. What it is immune to is
overfitting **in $B$**.

</details>

---

### Q6. NB-03 said prune your trees. This notebook says leave them unpruned. Which is right?

<details><summary>Answer</summary>

**Both, and the reason is Q2.**

**Alone:** a tree's error is bias + variance, and pruning trades a little bias for a large
variance reduction. That is the right trade when nothing else will control the variance.

**In a forest:** averaging already removes variance — that is its entire mechanism — but it
**cannot remove bias**. So you want base learners that are as low-bias as possible and let the
ensemble clean up the variance. Pruning would add bias the ensemble can never undo, while
removing variance it was going to remove for free.

Part 1.6 measures this: for the single tree, depth is a real trade-off with a peak; for the
forest, deeper is better or neutral all the way to unlimited.

**When to prune inside a forest anyway:**

- very noisy data, where deep trees memorise so much noise that even averaging cannot cope
  (`min_samples_leaf=5..20` is the usual lever)
- memory or latency limits — 500 unpruned trees on a million rows is a large object
- extremely high-dimensional data, where depth explodes

The general principle is worth carrying: **the right amount of regularisation for a base
learner depends on what the ensemble does with it.** Boosting takes this further still —
there the base learners are deliberately *stumps*, because boosting reduces bias and needs
variance kept in check.

</details>

---

### Q7. MDI says a feature has importance 0.15. Permutation importance says 0.002. Which is right?

<details><summary>Answer</summary>

**They are answering different questions and both may be correct.**

- **MDI** = total impurity reduction from splits on that feature, measured on **training**
  data. A feature can accumulate a lot of it by making splits that helped memorise training
  noise. MDI is also biased towards **high-cardinality and continuous** features, which offer
  more candidate thresholds and so more chances to look useful by luck (NB-03 Challenge 2).
- **Permutation** = how much the score drops on **held-out** data when the column is shuffled.
  It is not fooled by training noise.

**The usual resolution: trust permutation-on-test.** But before concluding the feature is
useless, check the other explanation:

**Correlated features.** If a redundant twin is still present, shuffling this column costs
nothing because the model reads the answer off the twin. Part 3 shows all three copies of one
driver getting *small* permutation importance individually, while dropping all three costs
0.22 AUC.

So the decision procedure is:

1. Is the feature correlated with others? → cluster and evaluate the **group**.
2. Not correlated, high MDI, ~0 permutation? → MDI is being fooled by training noise or
   cardinality. Trust permutation.
3. Need the answer for a real decision (drop a data feed)? → **drop-column**, the only method
   that answers "what if I genuinely no longer have this?".

</details>

---

### Q8. Three features are near-copies of the same underlying quantity. What does that do to each importance method, and what do you report?

<details><summary>Answer</summary>

- **Single-tree MDI:** essentially arbitrary. One copy wins the root split and takes almost
  all the credit; Part 3 shows 0.657 / 0.125 / 0.036 for three statistically identical
  columns. Refit on a resample and a different one wins (NB-03's instability).
- **Forest MDI:** spread **evenly** across the three (0.237 / 0.238 / 0.246), because each tree
  sees a different random feature subset so each copy gets its turn. This is a genuine
  improvement in honesty — but it also means the *group's* total importance is now split three
  ways, so each looks a third as important as the driver really is.
- **Permutation:** each individually small, because the other two cover for the shuffled one.
- **Drop-column, one at a time:** also small, for the same reason.
- **Drop-column, all three together:** enormous (−0.22 AUC in Part 3). This is the truth.

**What you report:** not per-feature numbers at all. Cluster the correlated features
(hierarchical clustering on Spearman correlation) and report importance **per group**. Then
say: *"the quantity these three columns measure is the dominant driver; which of the three you
keep is arbitrary."*

**The dangerous failure this prevents:** feature selection by "drop everything with low
importance". Applied naively, all three copies look droppable, and dropping them all destroys
the model.

</details>

---

### Q9. When is a random forest genuinely the wrong choice?

<details><summary>Answer</summary>

- **You need to extrapolate.** Every tree limitation from NB-03 Q8 survives averaging: a
  forest's prediction is an average of leaf values, so it can never leave the range of the
  training targets. Trending time series, physical laws, any extrapolation — use a linear
  component.
- **You need the model to be readable.** One tree can be printed; 500 cannot. If a rule set is
  the deliverable, an ensemble does not satisfy the requirement at any accuracy.
- **The relationship really is linear.** A forest approximates a smooth linear surface with
  axis-aligned boxes and needs a lot of data to do it well. NB-02 Part 2 shows gradient
  boosting *losing* to logistic regression on genuinely linear data — the same argument
  applies here.
- **You need calibrated probabilities out of the box.** A forest's `predict_proba` is a vote
  share, typically pulled toward the middle. Logistic regression is better calibrated by
  construction; otherwise wrap in `CalibratedClassifierCV`.
- **Very high-dimensional sparse data** (text, one-hot at scale). With 50,000 mostly-zero
  features, `sqrt(p)` random features rarely contains an informative one. Linear models
  dominate here.
- **Latency or memory are tight.** 500 deep trees is a large object with real inference cost.
- **Maximum tabular accuracy is the only goal.** Gradient boosting usually wins (NB-05),
  though at the price of hyperparameters that actually need tuning.

**Where a forest genuinely shines:** medium-sized tabular data, mixed feature types, non-linear
structure and interactions, minimal tuning, and a very strong out-of-the-box baseline.

</details>

---

### Q10. `ExtraTreesClassifier` picks split thresholds at random and often matches a random forest. How can that possibly work?

<details><summary>Answer</summary>

Because Q2's formula does not care how you achieve decorrelation, only that you do.

Extra Trees pushes randomisation one step further than `max_features`: for each candidate
feature it draws a **random threshold** rather than searching for the best one, and keeps the
best of those random candidates. (By default it also skips bootstrapping, using the whole
dataset for every tree.)

The effect on the two terms:

- $\sigma^2$ **rises** — random thresholds make each tree worse.
- $\rho$ **falls sharply** — the trees are far more different from each other.

Often the product $\rho\sigma^2$ falls, so the ensemble matches or beats a forest. It is also
**faster to fit**, because the exhaustive threshold search is the expensive part of tree
growing (Part 1.3 shows the fit-time difference).

**The general lesson, which is the useful one:** in a large ensemble, individual model quality
matters less than model *diversity*. That intuition is why so many ensembling tricks look like
deliberate sabotage of the base learner — random subspaces, random thresholds, dropout,
column subsampling in XGBoost. They are all attacks on $\rho$.

Extra Trees is not universally better: on very noisy data the random thresholds can cost too
much accuracy per tree. It costs one line to try.

</details>

---

### Q11. `predict_proba` on a forest returns 0.73. What is that number, and can you use it in an expected-value calculation?

<details><summary>Answer</summary>

It is the **mean of the individual trees' leaf class-proportions** — roughly "73% of the
forest's weighted vote is for class 1". It is *not* a fitted probability from a likelihood
model the way logistic regression's is.

**Is it calibrated?** Often only roughly. Forests are typically **under-confident** — pulled
toward the middle — because averaging many trees pulls extremes back. A prediction of 1.0
requires *every* tree to be unanimous. Niculescu-Mizil & Caruana (2005) measured exactly this
across model families.

Anything you did in NB-02 Part 3 also breaks it further: `class_weight`, resampling or SMOTE
distort the probabilities on top.

**So before any expected-value maths:**

1. Plot a **calibration curve** on held-out data (Foundations Part 7.4).
2. Check **Brier score** and **log-loss**, not just AUC — AUC cannot see calibration at all.
3. If it is off, wrap in `CalibratedClassifierCV` (`sigmoid` for small data, `isotonic` for
   large).

**If you are only ranking or thresholding**, calibration does not matter and the raw vote
share is fine. The distinction is exactly NB-02's: a *model* metric versus a *decision* metric.

</details>

---

### Q12. Your forest gets 0.94 CV AUC. Your colleague's gets 0.97 on the same data. Name five things that could explain the gap, in the order you would check them.

<details><summary>Answer</summary>

Deliberately in order of likelihood, and note that the first three mean their number is
**wrong**, not better:

1. **Leakage.** By far the most likely explanation for a surprising jump. A feature known only
   after the outcome, an ID correlated with the target, or preprocessing fitted before the
   split (Foundations Part 3). Ask what the top features are — a suspiciously dominant one is
   the tell.
2. **A different validation scheme.** Random `KFold` where the data has groups or time
   structure inflates scores dramatically (Foundations Part 3.3 measures +0.45 AUC for group
   leakage). Check `cv=` before anything else.
3. **Tuning on the test set.** If they tried thirty configurations and reported the best test
   score, that number includes the selection luck (Foundations Part 3.5).
4. **Genuinely better features.** Domain knowledge, ratios, aggregates, interactions. This is
   the legitimate and most valuable source of a real gap.
5. **Hyperparameters** — `max_features`, `min_samples_leaf`, `class_weight`, more trees.
   Real, but usually worth a point or two at most, not three.

**How to settle it:** run both models under the *same* CV object on the *same* splits, and
compare fold by fold, not just means. If their advantage survives your validation scheme, ask
which features they added — and audit those for #1.

The professional instinct to build: **a surprisingly good score is a bug report until proven
otherwise.**

</details>

---

## Coding challenges

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 1
# Implement bagging from scratch: bootstrap, fit, average.
#
# Then verify Q2's formula empirically - measure the actual variance of your
# ensemble's predictions across many independent training sets, for B = 1, 5,
# 25, 100, and check it falls the way rho*s^2 + (1-rho)*s^2/B predicts.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 1: one solution ------------------------------------------
def fit_bagged_trees(X, y, n_estimators=50, seed=0, max_features=None):
    """Bagging from scratch: bootstrap rows, fit a deep tree on each."""
    rng = np.random.default_rng(seed)
    n = len(X)
    trees = []
    for _ in range(n_estimators):
        idx = rng.integers(0, n, n)                     # bootstrap WITH replacement
        t = DecisionTreeRegressor(max_features=max_features,
                                  random_state=int(rng.integers(0, 1_000_000)))
        trees.append(t.fit(X[idx], y[idx]))
    return trees


def predict_bagged(trees, X):
    return np.mean([t.predict(X) for t in trees], axis=0)


# A fixed test point, and many independent training sets, so we can measure the
# variance of the ENSEMBLE's prediction directly.
def true_f(X):
    return np.sin(X[:, 0] * 2) * 3 + X[:, 1]


rng = np.random.default_rng(0)
x_query = np.array([[0.5, 0.5]])
n_train, n_replicates = 300, 120

print(f"{'B':>5} {'variance of prediction':>24} {'x reduction vs B=1':>21}")
print("-" * 54)
baseline_var = None
for B in [1, 5, 25, 100]:
    preds = []
    for r in range(n_replicates):
        Xt = rng.uniform(0, 1, size=(n_train, 2))
        yt = true_f(Xt) + rng.normal(0, 0.5, n_train)
        preds.append(predict_bagged(fit_bagged_trees(Xt, yt, B, seed=r), x_query)[0])
    v = float(np.var(preds))
    baseline_var = v if baseline_var is None else baseline_var
    print(f"{B:>5} {v:>24.5f} {baseline_var / v:>21.2f}")

print()
print("Variance falls steeply from B=1 to B=25 and then flattens - exactly the 1/B decay of")
print("the second term, running into the rho*sigma^2 floor set by the first.")
print()
print("Note it does NOT fall to zero. If the trees were independent (rho=0) it would keep")
print("falling; they share ~63% of their training rows, so rho > 0 and there is a floor.")

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 2
# Show that OOB error approximates cross-validation, and find where it breaks.
#
# Compare OOB score against 5-fold CV on i.i.d. data (they should agree), then
# on GROUPED data where the same entity appears many times (they should not).
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 2: one solution ------------------------------------------
from sklearn.model_selection import GroupShuffleSplit

# (a) i.i.d. data - OOB and CV should broadly agree
forest_iid = RandomForestClassifier(n_estimators=500, oob_score=True,
                                    random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_tr, yc_tr)
cv_iid = cross_val_score(RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE,
                                                n_jobs=-1),
                         Xc_tr, yc_tr, cv=StratifiedKFold(5, shuffle=True,
                                                          random_state=RANDOM_STATE)).mean()
print("(a) i.i.d. data")
print(f"    OOB score      {forest_iid.oob_score_:.4f}")
print(f"    5-fold CV      {cv_iid:.4f}")
print(f"    test score     {forest_iid.score(Xc_te, yc_te):.4f}")
print("    -> all three broadly agree; OOB is slightly pessimistic, as expected (Q4)")

# (b) grouped data - 50 customers, 40 applications each, label fixed per customer
rng = np.random.default_rng(5)
n_groups, per_group = 50, 40
group = np.repeat(np.arange(n_groups), per_group)
fingerprint = rng.normal(0, 1, size=(n_groups, 6))
Xg = fingerprint[group] + rng.normal(0, 0.3, size=(len(group), 6))
yg = rng.integers(0, 2, n_groups)[group]        # label is a property of the CUSTOMER

forest_g = RandomForestClassifier(n_estimators=500, oob_score=True,
                                  random_state=RANDOM_STATE, n_jobs=-1).fit(Xg, yg)
group_cv = cross_val_score(
    RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
    Xg, yg, groups=group,
    cv=GroupShuffleSplit(5, test_size=0.25, random_state=RANDOM_STATE)).mean()

print("\n(b) grouped data (same customer appears 40 times, label fixed per customer)")
print(f"    OOB score           {forest_g.oob_score_:.4f}   <- looks excellent")
print(f"    GroupShuffleSplit   {group_cv:.4f}   <- the honest number")
print(f"    optimism            {forest_g.oob_score_ - group_cv:+.4f}")
print()
print("OOB is catastrophically optimistic here. A row that is out-of-bag for a tree still")
print("has 39 SIBLINGS from the same customer in that tree's bag, so 'unseen' is a fiction -")
print("the tree has effectively memorised that customer.")
print()
print("OOB assumes i.i.d. rows, exactly like a random split does. On grouped, clustered or")
print("time-ordered data it is not just imprecise, it is WRONG - and unlike a random split,")
print("there is no `groups=` argument to fix it. You must use GroupKFold instead.")

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 3
# Reproduce the max_features trade-off curve from Part 1.3, but measure BOTH
# halves of Q2's formula:
#   - mean pairwise correlation between the trees (rho)
#   - mean accuracy of the INDIVIDUAL trees (a proxy for 1/sigma^2)
# and show the ensemble peaks where the product is best, not where either is.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 3: one solution ------------------------------------------
settings = [1, 2, 4, "sqrt", 8, 12, None]
rows = []
for mf in settings:
    f = RandomForestClassifier(n_estimators=120, max_features=mf,
                               random_state=RANDOM_STATE, n_jobs=-1).fit(Xr_tr, yr_tr)

    preds = np.array([t.predict_proba(Xr_te)[:, 1] for t in f.estimators_])
    c = np.corrcoef(preds)
    rho = (c.sum() - len(c)) / (len(c) * (len(c) - 1))

    # How good is a TYPICAL single tree in this forest?
    solo = np.mean([roc_auc_score(yr_te, p) for p in preds])
    rows.append({"max_features": str(mf), "tree_rho": rho,
                 "mean_solo_AUC": solo,
                 "ensemble_AUC": roc_auc_score(yr_te, f.predict_proba(Xr_te)[:, 1])})

trade = pd.DataFrame(rows)
print(trade.to_string(index=False, float_format=lambda v: f"{v:10.4f}"))

best = trade.loc[trade["ensemble_AUC"].idxmax()]
print(f"\nbest ensemble at max_features={best['max_features']}: "
      f"rho={best['tree_rho']:.3f}, solo AUC={best['mean_solo_AUC']:.3f}")
print(f"lowest correlation  at max_features={trade.loc[trade['tree_rho'].idxmin(), 'max_features']}")
print(f"strongest solo tree at max_features={trade.loc[trade['mean_solo_AUC'].idxmax(), 'max_features']}")

fig, ax1 = plt.subplots(figsize=(7.5, 4))
ax1.plot(trade["max_features"], trade["tree_rho"], marker="o", color="crimson",
         label="tree correlation (rho)")
ax1.plot(trade["max_features"], trade["mean_solo_AUC"], marker="s", color="seagreen",
         label="mean single-tree AUC")
ax1.set_xlabel("max_features"); ax1.set_ylabel("rho / solo AUC")
ax2 = ax1.twinx()
ax2.plot(trade["max_features"], trade["ensemble_AUC"], marker="^", color="steelblue",
         label="ENSEMBLE AUC")
ax2.set_ylabel("ensemble AUC")
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="center right")
plt.title("The ensemble peaks between the two extremes")
plt.tight_layout(); plt.show()

print()
print("The two EXTREMES both lose. max_features=1 gives the lowest tree correlation and the")
print("weakest individual trees; max_features=None gives the strongest trees and the highest")
print("correlation. Neither wins - the best settings sit in the middle of the range, which is")
print("what minimising rho*sigma^2 predicts.")
print()
print("Be careful how hard you read the exact argmax, though: the top few settings are within")
print("a few thousandths of each other, which is inside the run-to-run noise. The robust")
print("finding is the SHAPE of the curve - both ends are worse - not which middle value wins.")

---
# Part 5 - Five datasets to practise on

| # | Dataset | Rows × cols | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **Breast cancer** | 569 × 30 | Forest vs single tree; correlated importance | ★☆☆☆☆ |
| 2 | **Adult / census** | 48,842 × 14 | Mixed types, imbalance, importance at scale | ★★☆☆☆ |
| 3 | **Covertype** | 581,012 × 54 | Scale — where `n_jobs` and OOB earn their keep | ★★★☆☆ |
| 4 | **Bike sharing** | 17,379 × 12 | **Regression + the extrapolation trap** | ★★★★☆ |
| 5 | **Ames housing** | 1,460 × 79 | Regression, high cardinality, forest vs boosting | ★★★★★ |

In [ ]:
from sklearn.datasets import fetch_openml

CATALOGUE = [
    ("Breast cancer",  lambda: load_breast_cancer(as_frame=True)),
    ("Adult (census)", lambda: fetch_openml(name="adult", version=2, as_frame=True)),
    ("Bike sharing",   lambda: fetch_openml(name="Bike_Sharing_Demand", version=2,
                                            as_frame=True)),
    ("Ames housing",   lambda: fetch_openml(name="house_prices", version=1, as_frame=True)),
]

print(f"{'dataset':<18} {'rows':>7} {'cols':>6} {'NaN':>7} {'text':>6}  target")
print("-" * 66)
for label, loader in CATALOGUE:
    try:
        b = loader()
        Xd = b.data
        n_text = len(Xd.select_dtypes(exclude="number").columns)
        print(f"{label:<18} {Xd.shape[0]:>7} {Xd.shape[1]:>6} "
              f"{int(Xd.isna().sum().sum()):>7} {n_text:>6}  {b.target.name}")
    except Exception as exc:
        print(f"{label:<18} unavailable - {type(exc).__name__}: {str(exc)[:32]}")

print("\nCovertype is fetched with fetch_covtype() and is ~75 MB - deliberately not")
print("downloaded here. Uncomment below when you want it.")
print("    # from sklearn.datasets import fetch_covtype")
print("    # cov = fetch_covtype(as_frame=True)")

### 1. Breast cancer — the gentle start

1. Compare a tuned single tree (NB-03) against a default forest. Quantify the gap, and the
   interpretability you gave up for it.
2. The 30 features are three statistics (mean / error / worst) of ten measurements — heavily
   correlated **by construction**. Cluster them and report importance per group (Part 3).
3. Does `max_features` matter here? Sweep it and plot against tree correlation.
4. Check calibration (Q11). Is `predict_proba` usable for expected-value maths?

---

### 2. Adult / census — the realistic one

```python
bunch = fetch_openml(name="adult", version=2, as_frame=True)
```

1. One-hot the 8 categoricals inside a `ColumnTransformer`. Note what happens to
   `max_features="sqrt"` once 14 columns become ~100 — is $\sqrt{100}$ still sensible?
2. Compare against `HistGradientBoostingClassifier` with native `categorical_features=`.
3. Compute MDI, permutation and drop-column importance for `education-num`. Do they agree?
   (`education` and `education-num` encode the same thing — this is Part 3 on real data.)
4. Check whether error rates differ by `sex` and `race`, as in NB-02.

---

### 3. Covertype — scale

```python
from sklearn.datasets import fetch_covtype
cov = fetch_covtype(as_frame=True)     # ~581k rows, 54 features, 7 classes
```

1. Time a fit with `n_jobs=1` versus `n_jobs=-1`. This is where the embarrassingly-parallel
   property becomes real money.
2. At this size 5-fold CV is genuinely expensive. **Use OOB instead** for model selection and
   verify against a single held-out split at the end.
3. Sweep `n_estimators` from 10 to 500 and plot test accuracy. Confirm Q5: it flattens, it
   never turns down.
4. Measure memory: `sys.getsizeof` via `joblib.dump` to disk. How large is a 500-tree unpruned
   forest on 581k rows, and does that fit your deployment budget?

---

### 4. Bike sharing — regression, and the trap

```python
bunch = fetch_openml(name="Bike_Sharing_Demand", version=2, as_frame=True)
```

1. `RandomForestRegressor` with a random split. Note the score.
2. Split by **time** instead. It collapses — ridership grew over the period, and Q9/NB-03 Q8
   says a forest cannot predict above its training range.
3. Confirm it directly: compare `max(y_pred)` against `max(y_train)` and `max(y_test)`.
4. Fix it — detrend, or difference, or fit a linear trend and let the forest model the
   residual. Measure the improvement.
5. Note `max_features` defaults to `1.0` for regression, i.e. **plain bagging**. Lower it to
   0.3 and re-measure — this is often the biggest single win on regression forests.

---

### 5. Ames housing — regression at full messiness

```python
bunch = fetch_openml(name="house_prices", version=1, as_frame=True)
```

79 features, 43 categorical, 6,965 missing values.

1. Build the `ColumnTransformer`. Note that a forest needs **no scaling** — how much simpler
   is this pipeline than NB-01's?
2. Tune `max_features` and `min_samples_leaf` with `RandomizedSearchCV`. Which matters more?
3. Compare against gradient boosting. On Ames, boosting usually wins — by how much, and is it
   worth it?
4. Log-transform the target (as NB-01 Part 6 does) and re-measure. Does it help a forest as
   much as it helps a linear model? Explain why or why not.

---
# Part 6 - Reading the literature

## Start here

**1. [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324)** —
Leo Breiman, *Machine Learning* 45(1):5-32, 2001.
> The paper. It introduces `max_features`, proves the generalisation bound in terms of
> **tree strength and correlation** — the $\rho$ and $\sigma^2$ of Part 1.2 — and defines the
> OOB estimate. Unusually readable for a foundational paper, and section 2 is essentially
> this notebook's Part 1.

**2. [Bagging Predictors](https://link.springer.com/article/10.1007/BF00058655)** —
Leo Breiman, *Machine Learning* 24(2):123-140, 1996.
> The prequel. Read it for one specific idea: Breiman's argument about **stability**. Bagging
> helps unstable learners (trees, neural nets) and does nothing for stable ones (k-NN, linear
> models). That is Part 1.2's bias/variance result, stated five years earlier in different
> language.

**3. [Bias in Random Forest Variable Importance Measures](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-8-25)** —
Strobl, Boulesteix, Zeileis & Hothorn, *BMC Bioinformatics* 8:25, 2007. **Free.**
> The empirical takedown of MDI, and the reason Part 3 exists. Shows the cardinality and
> correlation biases with controlled simulations, and proposes conditional permutation
> importance as a fix. If you report feature importance to anyone, read this one.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.1 — the bootstrap | **Efron**, *Bootstrap Methods: Another Look at the Jackknife*, Ann. Statist. 7(1), **1979** | 🔍 |
| 1.2 — bagging, and the stability argument | **Breiman**, *Bagging Predictors*, **1996** | 🔍 |
| 1.2 — the variance formula | Hastie, Tibshirani & Friedman, *ESL* §15.2 — [free PDF](https://hastie.su.domains/ElemStatLearn/) | ✅ |
| 1.3 — decorrelation, `max_features`, OOB | **Breiman**, *Random Forests*, **2001** | 🔍 |
| 1.3 — random subspaces, independently | **Ho**, *The Random Subspace Method for Constructing Decision Forests*, IEEE TPAMI 20(8), **1998** | 🔍 |
| 1.3 — extremely randomised trees | **Geurts, Ernst & Wehenkel**, *Extremely Randomized Trees*, Machine Learning 63(1), **2006** | 🔍 |
| Part 3 — MDI bias | **Strobl et al.**, **2007** — [link](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-8-25) | ✅ |
| Part 3 — correlated features | **Strobl, Boulesteix, Kneib, Augustin & Zeileis**, *Conditional Variable Importance for Random Forests*, BMC Bioinformatics 9:307, **2008** — [link](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-9-307) | ✅ |
| Q11 — calibration by model family | **Niculescu-Mizil & Caruana**, ICML **2005** — [pdf](https://www.cs.cornell.edu/~alexn/papers/calibration.icml05.crc.rev3.pdf) | ✅ |
| Q9 — do we still need anything else? | **Fernández-Delgado et al.**, *Do we Need Hundreds of Classifiers to Solve Real World Classification Problems?*, JMLR 15, **2014** — [pdf](https://jmlr.org/papers/v15/delgado14a.html) | ✅ |
| Q9 — the modern counter | **Grinsztajn, Oyallon & Varoquaux**, *Why do tree-based models still outperform deep learning on tabular data?*, NeurIPS **2022** — [arXiv](https://arxiv.org/abs/2207.08815) | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com) — Breiman's papers are widely mirrored

### If you read only two

**Breiman (2001)** for what a forest *is*, and **Strobl et al. (2007)** for why the importance
numbers you are about to put in a slide may be wrong.

Then, for fun: **Fernández-Delgado et al. (2014)** benchmarked 179 classifiers across 121
datasets and found random forests the best family overall. It is a flawed study in several
ways — but it is the empirical reason "just try a random forest first" became standard advice,
and it is worth knowing where that advice came from.

---
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Fitting is very slow | Default `n_jobs=None` uses one core | **`n_jobs=-1`**, always |
| `UserWarning: Some inputs do not have OOB scores` | Too few trees for every row to be OOB somewhere | Raise `n_estimators` above ~50, or ignore for small forests |
| OOB much higher than test | Leakage, or grouped/time data (Challenge 2) | `GroupKFold` / `TimeSeriesSplit`; OOB is invalid there |
| OOB slightly lower than test | **Normal** (Q4) | Nothing — it is mildly pessimistic by design |
| Model file is enormous | 500 unpruned trees on many rows | `min_samples_leaf`, `max_depth`, fewer trees, or `HistGradientBoosting` |
| `predict_proba` never near 0 or 1 | Vote averaging pulls to the middle (Q11) | `CalibratedClassifierCV` if you need real probabilities |
| Regression predictions capped | Structural — cannot leave training target range (Q9) | Detrend/difference, or add a linear component |
| A noise feature has non-zero MDI | MDI is training-set impurity; never exactly zero | Permutation importance on held-out data |
| Importance spread thinly over several features | They are correlated (Part 3, Q8) | Cluster, then report per group |
| Forest barely beats one tree | Data is nearly linear, or trees are pruned too hard | Check against logistic regression; unprune (Q6) |
| Poor performance on wide sparse data | `sqrt(p)` rarely hits an informative feature (Q9) | Linear model, or raise `max_features` |

## Checklist for shipping a random forest

Everything in the Foundations checklist, plus:

- [ ] Is `n_jobs=-1` set?
- [ ] Is `n_estimators` as high as the budget allows (and did I confirm the curve flattened)?
- [ ] Did I tune `max_features` — especially for **regression**, where the default is all features?
- [ ] Are the base trees unpruned (or did I have a specific reason to prune)?
- [ ] Is OOB valid for this data — i.i.d. rows, no groups, no time order?
- [ ] Did I check calibration before letting anyone multiply `predict_proba` by a value?
- [ ] Did I cluster correlated features before reporting any importance?
- [ ] Am I using permutation-on-test, not MDI, for any claim about what matters?
- [ ] For regression: are production inputs inside the training target range?
- [ ] Did I compare against a single tree **and** a linear model, so I know what the ensemble bought?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `gradient_boosting_zero_to_hero.ipynb` | **The direct sequel.** The other way to combine trees: sequential error-correction rather than parallel averaging. Reduces *bias* where bagging reduces *variance* — and that one difference explains every behavioural contrast in Part 1.5 and Q5/Q6. |
| `explainable_ai_zero_to_hero.ipynb` | Part 3 is the appetiser. SHAP, partial dependence, and why explanation methods disagree. |
| `imbalanced_classification_zero_to_hero.ipynb` | `class_weight="balanced_subsample"` in Part 2, done properly. |

See [`ZERO_TO_HERO_PLAN.md`](../ZERO_TO_HERO_PLAN.md) for the full roster and status.